# Testing epsilon and gamma
We have two free parameters that we can play around to fit the data: the degree of abstraction ($\epsilon$) but also, the discount factor ($\gamma$). In all the previous notebooks, we have kept $\gamma$ constant. But it could be the case that participants actually employ a particular discount factor and rely on abstraction at the same time, which would entail that the best fit can be obtained by finding the best combination of the two. For that purpose, we test combinations of the two parameters:

## Prepare data for model fitting:

In [1]:
import numpy as np
import pandas as pd
import os
from scipy.stats import zscore
from stabst.utils import plot_state_matrix, beh_barplot, abstract2ground_value
from scipy.special import expit
from stabst.MarkovDecisionProcess import MDP
from stabst.TaskConfig import LimitedEnergyTask
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
import arviz as az
import pymc as pm

# ===================================================================================
# Prepare the data:
# Download data from Florian's repo:
data = pd.read_csv("../data/raw_data/all_participants_data.csv", on_bad_lines='skip')
# Remove nans:
data = data.dropna()
# Flip responses: 1 = accept:
data["response"] = np.abs(data["response"] - 1)
# Remove trials with max energy or too low energy:
data = data[data["energy"] != 6].reset_index(drop=True)
data = data[data["energy"] >= data["energy_cost"]].reset_index(drop=True)
# Remove the 
# Get subject IDs:
subj_idx_raw, subj_labels = pd.factorize(data["vpn"])
coords = {
    "observation": np.arange(len(data)),
    "subject": subj_labels,
}

transitions_costs = {
    0: [1, 1],
    1: [2, 1],
    2: [1, 2],
    3: [2, 2]
}
data["fc"] = [transitions_costs[row["transition"]][1] for _, row in data.iterrows()]

# Generate decision values associated with combinations of discount factors and abstraction levels:

In [2]:
# Create the task and its parameters (transition probability, reward...):
gammas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99]

# Loop through each gammas:
for gamma in gammas:
    task = LimitedEnergyTask(O=[1, 2, 3, 4], p_offer=[1/4] * 4)
    task.build()

    # Create full MDP and compute solution for later reference:
    gamma = 0.99
    task_mdp = MDP(task.states, task.tp, task.r, gamma, s2i=task.s2i)
    V_full, Q_full = task_mdp.backward_induction()
    R_full = task_mdp.expected_return(task_mdp.greedy_policy(Q_full), initial_energy=3)

    # Compute pairwise distances:
    if not os.path.isfile(f'../data/bids/limited_energy/derivatives/state_abstraction/bisimulation_distance_matrix_gamma_{gamma}.npy'):
        distances_matrix = task_mdp.bisim_metric(gamma=0.99, tol=1e-3, njobs=-1, max_iters=1000)
        distances_matrix = distances_matrix / np.max(distances_matrix)
        if not os.path.isdir("../data/bids/limited_energy/derivatives/state_abstraction"):
            os.makedirs("../data/bids/limited_energy/derivatives/state_abstraction")
        np.save(f'../data/bids/limited_energy/derivatives/state_abstraction/bisimulation_distance_matrix_gamma_{gamma}.npy', distances_matrix)
    else:
        distances_matrix = np.load(f'../data/bids/limited_energy/derivatives/state_abstraction/bisimulation_distance_matrix_gamma_{gamma}.npy')

    unique_dist = np.unique(np.round(distances_matrix, decimals=3))
    n_states = len(task_mdp.states)
    abstraction_distances = []
    DVs = []

    for eps in unique_dist: #unique_dist:
        # Generate abstracted MDP at that level of abstraction:
        abstract_mdp, state_classes, class_of_state = task_mdp.distance_reduce_mdp(eps + 0.0001, distance_matrix=distances_matrix)
        if n_states - len(abstract_mdp.states) < 10:
            continue
        else:
            n_states = len(abstract_mdp.states)
            abstraction_distances.append(eps)
        # Solve the MDP:
        V_R, Q_R = abstract_mdp.backward_induction()
        # Project back to Ground space:
        V_from_abstract, Q_from_abstract = abstract2ground_value(class_of_state, V_R, Q_R)
        dv = Q_from_abstract[:, 1] - Q_from_abstract[:, 0]
        # Add decision values to the data: 
        dv_trials = []
        mdl_sim_resps = []
        decision_match = []
        for trial_i, trial in data.iterrows():
            e, o, cc, t = trial.energy, trial.reward, trial.energy_cost, trial.trial
            fc = transitions_costs[trial.transition][1]
            dv_trials.append(dv[task.s2i[(e, o, cc, fc, t+1)]])
            # Simulate data according to that plan:
            mdl_sim_resps.append(np.random.binomial(1, expit(dv[task.s2i[(e, o, cc, fc, t+1)]])))
            decision_match.append(1 - np.abs(trial.response - mdl_sim_resps[-1]))
        data[f'DV-eps={eps}, gamma={gamma}'] = dv_trials

# Specify the model:

In [3]:
# Extract the data of one subject:
subj_idx_raw, subj_labels = pd.factorize(data["vpn"])
coords = {
    "subject": subj_labels,
    "coef": ["beta_plan", "beta_basic", "beta_maxE", "beta_mineE_LC", "beta_mineE_HC"],
}

with pm.Model(coords=coords) as hierarchical_binomial_regression:
    y_obs = pm.Data("y_obs", data["response"])
    dv = pm.Data("Decision values", data[f'DV-eps={0.0}, gamma={0.1}'])
    is_basic = pm.Data("is_basic", data["is_basic"])
    is_maxE = pm.Data("is_maxE", data["is_maxE"])
    is_minE_LC = pm.Data("is_minE_LC", data["is_minE_LC"])
    is_minE_HC = pm.Data("is_minE_HC", data["is_minE_HC"])
    subj_idx = pm.Data("subj_idx", subj_idx_raw)

    # Population level priors:
    beta_pop = pm.Normal("beta_pop", mu=0, sigma=2, dims="coef")
    sigma_pop = pm.HalfNormal('sigma_pop', sigma=2, dims="coef")
    # Non centered parametrization of within subject coefficients
    z = pm.Normal("z", 0, 1, dims=("subject", "coef"))
    beta_sub = pm.Deterministic("beta_sub", beta_pop + z * sigma_pop, dims=("subject","coef"))

    # Likelihood
    p = pm.Deterministic("p", 
                         pm.math.sigmoid(
                             beta_sub[subj_idx, 0] * dv,
                             beta_sub[subj_idx, 1] * is_basic,
                             beta_sub[subj_idx, 2] * is_maxE,
                             beta_sub[subj_idx, 3] * is_minE_LC,
                             beta_sub[subj_idx, 4] * is_minE_HC,
                             ))
    pm.Bernoulli("y", p=p, observed=y_obs)



KeyError: 'DV-eps=0.0, gamma=0.1'

## Fit model:

In [ ]:
# Model with each abstraction levels and sample:
traces = {}
for gamma in gammas:
    for eps in abstraction_distances:
        with hierarchical_binomial_regression:
            pm.set_data({"Decision values": data[f'DV-eps={eps}, gamma={gamma}']})
            traces[f'DV-eps={eps}, gamma={gamma}'] =  pm.sample(1000, tune=1000, target_acceptance=0.95, idata_kwargs={"log_likelihood": True})
            data[f'p(y=1|y_obs)@eps={eps}, gamma={gamma}'] = traces[f'DV, eps={eps}'].posterior.p.mean(dim=["chain", "draw"])

# Model comparison:

In [ ]:
# Compare models:
model_comparison = az.compare(traces)
az.plot_compare(model_comparison)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta_pop[intercept],0.752,0.135,0.513,1.011,0.004,0.002,926.0,1819.0,1.0
beta_pop[slope],5.593,0.290,5.063,6.137,0.007,0.005,1525.0,2019.0,1.0
sigma_pop[intercept],0.778,0.113,0.578,0.988,0.003,0.002,1443.0,2190.0,1.0
sigma_pop[slope],1.571,0.241,1.141,2.029,0.006,0.004,1630.0,2053.0,1.0
